In [ ]:
'''
    # 02 - Data Preprocessing Pipeline

    ## 1. Introduction & Objectifs
    - Résumé rapide de l’EDA (déséquilibre, tailles variables...)
    - Objectifs de ce notebook (redimensionnement, normalisation, augmentation)

    ## 2. Configuration & Chemins
    - Définition de `DATA_DIR`
    - Import des bibliothèques (tensorflow.keras, albumentations, etc.)

    ## 3. Calcul des Class Weights
    - Calcul automatique à partir des nombres du train (1341 Normal / 3875 Pneumonia)
    - Explication pourquoi on les utilise

    ## 4. Pipeline de Prétraitement
    - Redimensionnement à 224×224
    - Normalisation (1./255 ou ImageNet statistics ?)
    - Création des générateurs de données

    ## 5. Data Augmentation
    - Configuration de l’augmentation
    - **Stratégie différenciée** : augmentation plus forte sur la classe NORMAL
    - Visualisation d’exemples d’images augmentées (très important pour comprendre)

    ## 6. Création des Data Generators / tf.data Pipeline
    - `ImageDataGenerator` (Keras) ou Albumentations + tf.data
    - Flow from directory avec class_mode='binary'

    ## 7. Test du pipeline
    - Vérifier la shape des batches (32, 224, 224, 3) ou (32, 224, 224, 1)
    - Afficher quelques images augmentées

    ## 8. Conclusion & Décisions
    - Pipeline prêt pour l’entraînement
    - Choix techniques retenus
    - Prochaines étapes (modélisation)

'''

In [ ]:
# ================================================
# 1 Introduction & Objectifs
# ================================================
'''
    EDA recap : 
        1 - imbalance of samples sizes of classes ['Normal', 'Pneumonia' ] in train split
        2 - diffrence : (1341 Normal / 3875 Pneumonia),  Pneumonia 3 times size of Normal
        
    goals of this norbook :
        1 - resizing
        2 - normalization
        3 - scaling
'''

In [ ]:
import tensorflow.keras as tk
import albumentations

# ================================================
# 2. Calcul des Class Weights
# ================================================

DATA_DIR = "../data/raw/chest_xray"

# Nombres issus de l'EDA (Phase 1)
train_normal     = 1341
train_pneumonia  = 3875
train_total      = train_normal + train_pneumonia
num_classes      = 2

print(f"Train set composition:")
print(f"   Normal    : {train_normal} images ({train_normal/train_total*100:.2f}%)")
print(f"   Pneumonia : {train_pneumonia} images ({train_pneumonia/train_total*100:.2f}%)")
print(f"   Total     : {train_total} images\n")

# ====================== CLASS WEIGHTS ======================
# Formule standard pour compenser le déséquilibre :
# weight_for_class = total_samples / (n_classes * samples_in_class)

weight_normal = train_total / (num_classes * train_normal)
weight_pneumonia = train_total / (num_classes * train_pneumonia)

print("=== Class Weights Calculation ===")
print(f"Weight for NORMAL    : {weight_normal:.4f}")
print(f"Weight for PNEUMONIA : {weight_pneumonia:.4f}")
print(f"Ratio (Normal / Pneumonia) : {weight_normal / weight_pneumonia:.2f}x\n")

# ====================== EXPLICATION ======================
"""
Pourquoi utilisons-nous les class weights ?

Le dataset d'entraînement est fortement déséquilibré (74.29% Pneumonia vs 25.71% Normal).
Sans compensation, le modèle va minimiser la loss principalement sur la classe majoritaire 
(Pneumonia), ce qui risque de biaiser les prédictions et de réduire le recall sur la classe Normal.

Les class weights permettent de donner plus d'importance aux erreurs commises sur la classe 
minoritaire (Normal) pendant le calcul de la fonction de perte.

Formule utilisée :
    weight_class_i = total_samples / (n_classes * n_samples_class_i)

Nous combinerons cette technique avec une data augmentation plus forte sur la classe Normal.
"""

# ====================== UTILISATION FUTURE ======================
"""
Comment utiliser ces poids pendant l'entraînement ?

Dans Keras/TensorFlow, nous passerons ce dictionnaire au paramètre class_weight :

    class_weight = {0: weight_normal, 1: weight_pneumonia}

Exemple dans model.fit():
    model.fit(..., class_weight=class_weight, ...)
"""





